# Google Colab — tái lập kết quả Loan Default Risk

Notebook này chạy **trực tiếp code trong `src/`**, không sao chép pipeline sang notebook. Vì vậy temporal split, feature engineering, calibration, threshold và seed luôn đồng nhất với dự án.

Chuẩn bị hai tệp:
1. File ZIP của thư mục `loan-default-risk-prediction` (có thể không chứa dữ liệu).
2. `lendingclub_2007_2011.csv`.

> Lưu ý: huấn luyện có 5-fold CV, Random Forest và calibration nên có thể mất vài phút trên Colab CPU.

In [ ]:
# Cố định đúng môi trường đã dùng để tạo artifact hiện tại.
%pip install -q pandas==2.2.3 numpy==2.1.3 scikit-learn==1.9.0 joblib==1.4.2 matplotlib==3.9.2 seaborn==0.13.2

import os
os.environ['PYTHONHASHSEED'] = '42'
print('Đã cài đặt môi trường tái lập.')

## 1. Tải mã nguồn dự án lên Colab

Nén thư mục dự án thành ZIP rồi chọn file ZIP khi ô tải tệp xuất hiện. Nếu thư mục dự án đã tồn tại trong `/content`, cell sẽ không yêu cầu tải lại.

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

CONTENT = Path('/content')
PROJECT = CONTENT / 'loan-default-risk-prediction'

if not PROJECT.exists():
    uploaded = files.upload()
    zip_files = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_files) != 1:
        raise ValueError('Hãy tải đúng một file ZIP của dự án.')
    archive = CONTENT / zip_files[0]
    shutil.unpack_archive(archive, CONTENT)

    # Hỗ trợ ZIP có thêm một thư mục bao ngoài.
    if not PROJECT.exists():
        candidates = [p for p in CONTENT.rglob('src/train.py') if p.parent.parent.is_dir()]
        if len(candidates) != 1:
            raise FileNotFoundError('Không xác định được thư mục gốc chứa src/train.py.')
        PROJECT = candidates[0].parent.parent

assert (PROJECT / 'src' / 'train.py').exists(), 'Thiếu src/train.py trong ZIP.'
print('Thư mục dự án:', PROJECT)

## 2. Cung cấp dữ liệu

Dữ liệu được đặt đúng tại `data/raw/lendingclub_2007_2011.csv`. Notebook không tải dữ liệu từ nguồn không xác minh và không tự ý phân phối lại dataset.

In [ ]:
DATA_PATH = PROJECT / 'data' / 'raw' / 'lendingclub_2007_2011.csv'
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
    if len(csv_files) != 1:
        raise ValueError('Hãy tải đúng một file lendingclub_2007_2011.csv.')
    shutil.move(str(CONTENT / csv_files[0]), DATA_PATH)

print(f'Dữ liệu: {DATA_PATH} ({DATA_PATH.stat().st_size / 1_000_000:.1f} MB)')

## 3. Kiểm tra dữ liệu và temporal split

Kết quả chuẩn: tổng 38.577 khoản vay có nhãn; train 18.061, validation 9.015 và test 11.501.

In [ ]:
import sys
import pandas as pd

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.data import load_data, temporal_split
from src.features import TARGET, create_target

labeled = create_target(load_data(DATA_PATH))
train_df, validation_df, test_df = temporal_split(labeled)
split_rows = {
    'total': len(labeled),
    'train': len(train_df),
    'validation': len(validation_df),
    'test': len(test_df),
}
expected_rows = {'total': 38577, 'train': 18061, 'validation': 9015, 'test': 11501}
assert split_rows == expected_rows, f'Số dòng khác dữ liệu chuẩn: {split_rows}'

pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(train_df), len(validation_df), len(test_df)],
    'default_rate': [train_df[TARGET].mean(), validation_df[TARGET].mean(), test_df[TARGET].mean()],
})

## 4. Huấn luyện toàn bộ pipeline

Cell này gọi đúng `src.train.train`, tạo artifact Colab riêng và xuất lại toàn bộ báo cáo.

In [ ]:
from src.train import RANDOM_STATE, train

COLAB_ARTIFACT = PROJECT / 'artifacts' / 'loan_default_colab.joblib'
COLAB_REPORTS = PROJECT / 'reports' / 'colab_reproduction'

artifact = train(
    data_path=DATA_PATH,
    output_path=COLAB_ARTIFACT,
    report_dir=COLAB_REPORTS,
    random_state=RANDOM_STATE,
)
artifact['model_name'], artifact['threshold'], artifact['metrics']

## 5. Xác nhận kết quả khớp dự án

Các metric được so với giá trị đã công bố. Dung sai `5e-4` chỉ bao phủ sai số làm tròn bốn chữ số; threshold và số dòng phải khớp chính xác.

In [ ]:
EXPECTED = {
    'pr_auc': 0.3384019475,
    'roc_auc': 0.7194800011,
    'f1': 0.3990726942,
    'recall': 0.6282586027,
    'precision': 0.2924047561,
    'balanced_accuracy': 0.6619848789,
    'brier_score': 0.1286475379,
}
TOLERANCE = 5e-4

rows = []
for metric, expected in EXPECTED.items():
    actual = float(artifact['metrics'][metric])
    difference = abs(actual - expected)
    rows.append({
        'metric': metric,
        'kết quả Colab': actual,
        'kết quả dự án': expected,
        'sai lệch tuyệt đối': difference,
        'khớp': difference <= TOLERANCE,
    })

comparison = pd.DataFrame(rows)
assert artifact['threshold'] == 0.14, f"Threshold bị lệch: {artifact['threshold']}"
expected_artifact_rows = {key: expected_rows[key] for key in ('train', 'validation', 'test')}
assert artifact['split_rows'] == expected_artifact_rows, f"Split bị lệch: {artifact['split_rows']}"
assert comparison['khớp'].all(), comparison.loc[~comparison['khớp']]
print('✅ Kết quả Colab khớp dự án trong dung sai làm tròn.')
comparison

## 6. Tải kết quả về máy

Artifact được tạo trong Colab và toàn bộ CSV/JSON đánh giá được đóng gói thành ZIP.

In [ ]:
shutil.copy2(COLAB_ARTIFACT, COLAB_REPORTS / COLAB_ARTIFACT.name)
RESULT_ZIP = shutil.make_archive(
    str(CONTENT / 'loan_default_colab_results'),
    'zip',
    root_dir=PROJECT,
    base_dir='reports/colab_reproduction',
)
print('Artifact:', COLAB_ARTIFACT)
files.download(RESULT_ZIP)